# 02_Finetune_Gemma3_Multimodal_From_HF_Dataset

이 노트북은 Hugging Face에 업로드한 통합 데이터셋을 내려받아,
Gemma 3 멀티모달 포맷으로 변환한 뒤 QLoRA로 학습합니다.

- 텍스트-only 샘플도 같은 모델로 학습
- 이미지+텍스트 샘플도 같은 모델로 학습
- 배치 크기는 기본적으로 1로 두어 모달 혼합 충돌을 줄입니다.

In [ ]:
# ============================================================
# 라이브러리 임포트 (최상단 통합)
# ============================================================

# === 표준 라이브러리 ===
import os
import random
from datetime import datetime

# === 데이터 처리 ===
import pandas as pd
from datasets import load_dataset

# === 딥러닝 ===
import torch
from peft import LoraConfig
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

# === 시각화 ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# === 이미지 처리 ===
from PIL import Image

# === 환경변수 / HuggingFace ===
from dotenv import load_dotenv
import huggingface_hub

# === DB 연결 ===
import mysql.connector

In [ ]:
def setKoreanFont():
    """matplotlib 한글 폰트 설정 - 깨짐 방지 (Windows: 맑은 고딕 우선 탐색)"""
    fontCandidates = ['Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'AppleGothic', 'DejaVu Sans']
    allFonts = fm.fontManager.ttflist
    availableFonts = []
    for i in range(0, len(allFonts)):
        availableFonts.append(allFonts[i].name)

    selectedFont = None
    for i in range(0, len(fontCandidates)):
        if fontCandidates[i] in availableFonts:
            selectedFont = fontCandidates[i]
            break

    if selectedFont is None:
        selectedFont = 'DejaVu Sans'
        print(f'[경고] 한글 폰트를 찾지 못함, 대체 사용: {selectedFont}')
    else:
        print(f'한글 폰트 설정 완료: {selectedFont}')

    matplotlib.rc('font', family=selectedFont)
    matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

setKoreanFont()

In [ ]:
# .env 파일 로드 (없어도 에러 없이 False 반환)
load_dotenv()

# HuggingFace 토큰
hf_token = os.getenv("HF_TOKEN", "")

In [3]:
huggingface_hub.login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# ===== 시드 고정 =====
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print('GPU =', torch.cuda.get_device_name(0))
print('torch =', torch.__version__)

# ===== HuggingFace 데이터셋 / 베이스 모델 =====
DATASET_REPO = os.getenv("DATASET_REPO", "hyokwan/multi_modal_sample")
BASE_MODEL   = os.getenv("BASE_MODEL",   "google/gemma-3-4b-it")

# ===== 학습 출력 경로 =====
OUTPUT_BASE_DIR = os.getenv("OUTPUT_BASE_DIR", "./models")
OUTPUT_DIR      = os.path.join(
    OUTPUT_BASE_DIR,
    f"gemma3_multimodal_lora_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
TENSORBOARD_DIR = os.path.join(OUTPUT_DIR, "tensorboard")

# ===== 데이터 샘플 수 제한 =====
_trainSamples     = os.getenv("MAX_TRAIN_SAMPLES", "")
_evalSamples      = os.getenv("MAX_EVAL_SAMPLES",  "")
MAX_TRAIN_SAMPLES = int(_trainSamples) if _trainSamples.strip() else None
MAX_EVAL_SAMPLES  = int(_evalSamples)  if _evalSamples.strip()  else None

# ===== 주요 튜닝 하이퍼파라미터 (.env 값 변경으로 시뮬레이션 가능) =====
MAX_SEQ_LENGTH              = int(os.getenv("MAX_SEQ_LENGTH",              768))
PER_DEVICE_TRAIN_BATCH_SIZE = int(os.getenv("PER_DEVICE_TRAIN_BATCH_SIZE", 1))
PER_DEVICE_EVAL_BATCH_SIZE  = int(os.getenv("PER_DEVICE_EVAL_BATCH_SIZE",  1))
GRAD_ACCUM                  = int(os.getenv("GRAD_ACCUM",                  4))
NUM_EPOCHS                  = int(os.getenv("NUM_EPOCHS",                  5))
LEARNING_RATE               = float(os.getenv("LEARNING_RATE",             2e-4))
LOGGING_STEPS               = int(os.getenv("LOGGING_STEPS",               10))
SAVE_STEPS                  = int(os.getenv("SAVE_STEPS",                  100))

# ===== LoRA 하이퍼파라미터 =====
LORA_R       = int(os.getenv("LORA_R",       16))
LORA_ALPHA   = int(os.getenv("LORA_ALPHA",   32))
LORA_DROPOUT = float(os.getenv("LORA_DROPOUT", 0.05))

print(f"데이터셋     : {DATASET_REPO}")
print(f"베이스 모델  : {BASE_MODEL}")
print(f"출력 경로    : {OUTPUT_DIR}")
print(f"TensorBoard  : {TENSORBOARD_DIR}")
print(f"에폭         : {NUM_EPOCHS}  |  LR: {LEARNING_RATE}  |  LoRA r/α: {LORA_R}/{LORA_ALPHA}")

### 데이터셋 불러오기

In [6]:
ds = load_dataset(DATASET_REPO)

train_ds = ds['train']

# test가 있는지 먼저 확인
try:
    eval_ds = ds['test']
except KeyError:
    eval_ds = ds['validation']

if MAX_TRAIN_SAMPLES:
    train_ds = train_ds.select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))
if MAX_EVAL_SAMPLES:
    eval_ds = eval_ds.select(range(min(MAX_EVAL_SAMPLES, len(eval_ds))))

print(train_ds)
print(eval_ds)
print(train_ds[0])

Dataset({
    features: ['modality', 'image', 'instruction', 'input', 'output', 'source', 'label'],
    num_rows: 55
})
Dataset({
    features: ['modality', 'image', 'instruction', 'input', 'output', 'source', 'label'],
    num_rows: 11
})
{'modality': 'text_only', 'image': None, 'instruction': '다음 운동 동작을 쉽게 설명하시오.', 'input': '레그 레이즈', 'output': '누워서 다리를 들어 올려 복부를 강화하는 운동이다.', 'source': 'generated', 'label': ''}


# 3. 환경 및 최적화 설정 (4비트 양자화)

In [7]:
# 현재 사용 중인 GPU의 주요 아키텍처 버전을 반환 8버전 이상 시 bfloat16 활용
# NVIDIA Ampere 아키텍처 이상 시에만 처리
if torch.cuda.get_device_capability()[0] >= 8:
    # 고속 attention 메커니즘을 구현하는 라이브러리
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# BitsAndBytesConfig 객체활용 양자화 설정
quant_config = BitsAndBytesConfig(
    # 모델을 4비트 양자화하여 로드할지 여부 결정
    load_in_4bit=True,
    # 양자화 방법 (nf4: Non-Uniform Quantization, "nf4","fp16 등))
    bnb_4bit_quant_type="nf4",
    # (4비트 양자화 시 사용할 데이터 타입, "torch.float16, bfloat16, float32 등)
    bnb_4bit_compute_dtype=torch_dtype,
    # 이중 양자화 사용여부 (이중 양자화는 양자화 과정에서 정밀도 높이기 위해 활용, 대신 더 연산은 복잡)
    bnb_4bit_use_double_quant=False
)

# 4. 베이스 모델 불러오기

In [9]:
# 멀티모달 모델 로드 (이미지 + 텍스트 입력 처리 가능)
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"":0},
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
    trust_remote_code=True,
    token = hf_token
)

# 멀티모달 모델 로드 (이미지 + 텍스트 입력 처리 가능)
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True,token = hf_token)

# processor 내부에서 tokenizer만 따로 사용
tokenizer = processor.tokenizer
# pad_token 없으면 eos_token으로 설정 # 학습 시 길이 같아야 함 문장1-안녕하세요 문장2 안녕 -> 안녕[pad][pad]
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

print('pad_token_id =', tokenizer.pad_token_id)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


pad_token_id = 0


In [ ]:
SYSTEM_MESSAGE = '당신은 멀티모달 도우미입니다. 텍스트 질문에는 정확히 답하고, 이미지가 주어지면 이미지를 보고 답하세요.'

def buildMessages(example):
    """샘플 데이터를 모달리티(텍스트/이미지+텍스트)에 따라 모델 메시지 구조로 변환"""
    instruction = str(example.get('instruction', '')).strip()
    userInput   = str(example.get('input', '')).strip()
    output      = str(example.get('output', '')).strip()
    modality    = str(example.get('modality', 'text_only')).strip()

    if instruction and userInput:
        userText = f"{instruction}\n\n[입력]\n{userInput}"
    else:
        userText = instruction or userInput

    if not userText:
        userText = '주어진 정보를 바탕으로 답하세요.'

    userContent = []

    if modality == 'image_text' and example.get('image') is not None:
        userContent.append({'type': 'image', 'image': example['image']})

    userContent.append({'type': 'text', 'text': userText})

    messages = [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_MESSAGE}]},
        {'role': 'user',      'content': userContent},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': output}]},
    ]
    return messages

In [11]:
# 소규모 샘플 확인
sample_rows = []
for ex in train_ds.select(range(min(4, len(train_ds)))):
    sample_rows.append({
        'modality': ex['modality'],
        'instruction': ex['instruction'][:80],
        'input': ex['input'][:80],
        'output': ex['output'][:80],
        'label': ex['label'],
        'image_present': ex['image'] is not None,
    })
pd.DataFrame(sample_rows)

,modality,instruction,input,output,label,image_present
0,text_only,다음 운동 동작을 쉽게 설명하시오.,레그 레이즈,누워서 다리를 들어 올려 복부를 강화하는 운동이다.,,False
1,text_only,운동 루틴을 간단히 구성하시오.,초보자 10분,"스트레칭, 브릿지, 플랭크 순으로 10분 루틴을 구성한다.",,False
2,image_text,이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.,동작명과 간단한 설명을 한국어로 답하세요.,동작명: bridge\n설명: 이 이미지는 bridge 동작 예시입니다.,bridge,True
3,image_text,이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.,동작명과 간단한 설명을 한국어로 답하세요.,동작명: plank\n설명: 이 이미지는 plank 동작 예시입니다.,plank,True


In [ ]:
def collate_fn(examples):
    """텍스트 + 이미지 데이터를 모델 학습용 배치로 변환 및 불필요 토큰 loss 제외"""
    texts        = []
    images       = []
    hasAnyImage  = False

    for i in range(0, len(examples)):
        example  = examples[i]
        messages = buildMessages(example)

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

        exampleImages = []
        userContent   = messages[1]["content"]
        for j in range(0, len(userContent)):
            if userContent[j]["type"] == "image":
                exampleImages.append(userContent[j]["image"])

        if len(exampleImages) > 0:
            hasAnyImage = True
            images.append(exampleImages)
        else:
            images.append([])

    if hasAnyImage:
        batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    else:
        batch = processor(text=texts, return_tensors="pt", padding=True)

    labels       = batch["input_ids"].clone()
    padTokenId   = processor.tokenizer.pad_token_id
    if padTokenId is not None:
        labels[labels == padTokenId] = -100

    imageTokenId = getattr(model.config, "image_token_id", None)
    if imageTokenId is not None:
        labels[labels == imageTokenId] = -100

    labels[labels == 262144] = -100
    batch["labels"] = labels
    return batch

## 5. LoRA 및 Trainer 설정

In [ ]:
# LoRA 설정 (.env 값 사용)
peft_params = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

# SFT 설정 (TensorBoard 활성화)
sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    logging_dir=TENSORBOARD_DIR,            # TensorBoard 로그 저장 경로

    # (A) 학습량
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,

    # (B) 입력 길이
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,

    # (C) 학습 파라미터
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",

    # (D) 로그 / 평가 / 저장
    logging_steps=LOGGING_STEPS,
    report_to='tensorboard',                # TensorBoard 리포트 활성화
    eval_strategy='no',
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,

    # (E) 혼합 정밀도
    bf16=torch.cuda.is_available(),

    # (F) 멀티모달 필수
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=None,
    peft_config=peft_params,
    processing_class=processor,
    data_collator=collate_fn,
    args=sft_args,
)

print(f"TensorBoard 로그 경로: {TENSORBOARD_DIR}")
print("tensorboard --logdir", TENSORBOARD_DIR)

In [ ]:
train_result = trainer.train()

In [ ]:
def plotTrainingLoss(trainer, outputDir):
    """학습 손실(Loss) 추이를 시각화하고 이미지로 저장"""
    rawHistory  = trainer.state.log_history
    logHistory  = []
    for i in range(0, len(rawHistory)):
        if 'loss' in rawHistory[i]:
            logHistory.append(rawHistory[i])

    if len(logHistory) == 0:
        print("학습 로그가 없어 그래프를 그릴 수 없습니다.")
        return

    steps  = []
    losses = []
    for i in range(0, len(logHistory)):
        steps.append(logHistory[i]['step'])
        losses.append(logHistory[i]['loss'])

    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, marker='o', linewidth=2, markersize=4, color='steelblue')
    plt.title('학습 손실 추이 (Training Loss)', fontsize=14)
    plt.xlabel('스텝 (Step)', fontsize=12)
    plt.ylabel('손실 (Loss)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    savePath = os.path.join(outputDir, 'training_loss.png')
    os.makedirs(outputDir, exist_ok=True)
    plt.savefig(savePath, dpi=150)
    plt.show()
    print(f"그래프 저장 완료: {savePath}")

plotTrainingLoss(trainer, OUTPUT_DIR)

In [ ]:
def saveTrainingResultToDB(trainOutput, modelName, datasetRepo, outputDir, tensorboardDir, trainConfig):
    """학습 결과를 MySQL DB에 저장 (실행 시간, 모델명, 저장 경로, 성능 지표 포함)"""
    try:
        conn = mysql.connector.connect(
            host=os.getenv("DB_HOST", "localhost"),
            port=int(os.getenv("DB_PORT", 3306)),
            database=os.getenv("DB_NAME", ""),
            user=os.getenv("DB_USER", ""),
            password=os.getenv("DB_PASSWORD", "")
        )
        cursor = conn.cursor()

        # 테이블 없으면 자동 생성
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS training_runs (
                id                    INT AUTO_INCREMENT PRIMARY KEY,
                run_timestamp         DATETIME      NOT NULL,
                model_name            VARCHAR(255),
                dataset_repo          VARCHAR(255),
                output_dir            VARCHAR(512),
                tensorboard_dir       VARCHAR(512),
                train_loss            FLOAT,
                train_runtime_sec     FLOAT,
                train_samples_per_sec FLOAT,
                global_step           INT,
                num_epochs            INT,
                learning_rate         FLOAT,
                lora_r                INT,
                lora_alpha            INT,
                lora_dropout          FLOAT,
                max_seq_length        INT,
                grad_accum            INT,
                batch_size            INT
            )
        """)

        metrics = trainOutput.metrics
        cursor.execute(
            """
            INSERT INTO training_runs (
                run_timestamp, model_name, dataset_repo, output_dir, tensorboard_dir,
                train_loss, train_runtime_sec, train_samples_per_sec, global_step,
                num_epochs, learning_rate, lora_r, lora_alpha, lora_dropout,
                max_seq_length, grad_accum, batch_size
            ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """,
            (
                datetime.now(),
                modelName,
                datasetRepo,
                outputDir,
                tensorboardDir,
                metrics.get('train_loss'),
                metrics.get('train_runtime'),
                metrics.get('train_samples_per_second'),
                trainOutput.global_step,
                trainConfig['num_epochs'],
                trainConfig['learning_rate'],
                trainConfig['lora_r'],
                trainConfig['lora_alpha'],
                trainConfig['lora_dropout'],
                trainConfig['max_seq_length'],
                trainConfig['grad_accum'],
                trainConfig['batch_size'],
            )
        )
        conn.commit()
        cursor.close()
        conn.close()

        trainLoss    = metrics.get('train_loss', 0)
        trainRuntime = metrics.get('train_runtime', 0)
        print("====== DB 저장 완료 ======")
        print(f"  모델명       : {modelName}")
        print(f"  데이터셋     : {datasetRepo}")
        print(f"  출력 경로    : {outputDir}")
        print(f"  TensorBoard  : {tensorboardDir}")
        print(f"  학습 손실    : {trainLoss:.4f}")
        print(f"  학습 시간    : {trainRuntime:.1f}초  ({trainRuntime/60:.1f}분)")
        print(f"  Global Step  : {trainOutput.global_step}")

    except Exception as e:
        print({"success": False, "message": str(e)})


trainConfig = {
    'num_epochs':     NUM_EPOCHS,
    'learning_rate':  LEARNING_RATE,
    'lora_r':         LORA_R,
    'lora_alpha':     LORA_ALPHA,
    'lora_dropout':   LORA_DROPOUT,
    'max_seq_length': MAX_SEQ_LENGTH,
    'grad_accum':     GRAD_ACCUM,
    'batch_size':     PER_DEVICE_TRAIN_BATCH_SIZE,
}

saveTrainingResultToDB(
    train_result,
    BASE_MODEL,
    DATASET_REPO,
    OUTPUT_DIR,
    TENSORBOARD_DIR,
    trainConfig
)

In [21]:
# 학습된 LoRA 어댑터(모델 가중치) 저장
# → 전체 모델이 아니라 "변경된 부분(LoRA)"만 저장됨
trainer.model.save_pretrained(OUTPUT_DIR)

# 멀티모달 전처리기 저장 (tokenizer + image processor 포함)
# → 추론 시 동일한 방식으로 입력 처리하기 위해 필요
processor.save_pretrained(OUTPUT_DIR)

# 저장 완료 로그 출력
print('adapter saved to', OUTPUT_DIR)

adapter saved to ./models/gemma3_multimodal_lora_output_20260607_214638


In [ ]:
def infer_unified(question, image=None, system=SYSTEM_MESSAGE, max_new_tokens=128, temperature=0.2):
    """텍스트 또는 이미지+텍스트 입력을 받아 모델 추론 결과를 반환"""
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': system}]},
        {'role': 'user',   'content': []},
    ]

    if image is not None:
        messages[1]['content'].append({'type': 'image', 'image': image})

    messages[1]['content'].append({'type': 'text', 'text': question})

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    if image is not None:
        inputs = processor(text=[prompt], images=[[image]], return_tensors='pt', padding=True)
    else:
        inputs = processor(text=[prompt], return_tensors='pt', padding=True)

    inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
        )

    answer_ids = outputs[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(answer_ids, skip_special_tokens=True)[0].strip()

In [23]:
# 텍스트-only 테스트
text_answer = infer_unified(
    question='금융 용어를 쉽게 설명하시오.\n\n[입력]\n기준금리',
    system='당신은 금융 개념을 쉽게 설명하는 도우미입니다.',
    max_new_tokens=96,
    temperature=0.0,
)
print(text_answer)

c:\Users\hkcode\llm_multimodal\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\hkcode\llm_multimodal\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\hkcode\llm_multimodal\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\hkcode\llm_multimodal\.venv\Lib\site-package

기준금리는 금융 시장의 기준이 되는 금리입니다. 은행들이 돈을 빌려주거나 예금을 받는 시세를 결정하는 데 영향을 미칩니다.


In [24]:
# 텍스트-only 테스트
text_answer = infer_unified(
    question='다음 문장을 코칭 스타일로 바꾸시오.\n\n[입력]\n허리를 펴세요',
    system='당신은 AI 코칭 도우미입니다.',
    max_new_tokens=96,
    temperature=0.0,
)
print(text_answer)

허리를 곧게 세우고 코어에 힘을 주세요.


In [ ]:
# 이미지 로드
image = Image.open("test_001.png").convert("RGB")

# 추론 실행
result = infer_unified("이 이미지에 무슨 동작 인가요?", image=image)
print(result)